# 04 — Évaluation Finale et Analyse des Compromis

**Projet : Optimisation multi-objectifs pour AutoML — Option 2**

Ce notebook :
1. Réentraîne le meilleur modèle optimisé (Random Forest avec les meilleurs hyperparamètres)
2. Compare baseline vs modèle optimisé sur toutes les métriques
3. Génère la matrice de confusion et le rapport de classification
4. Analyse critique des 4 méthodes d'optimisation
5. Recommandations par contexte d'usage

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
import os
import pickle
import tracemalloc
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    f1_score
)

print("Librairies chargées.")

## 1. Chargement des données et des résultats d'optimisation

In [ ]:
X_train = np.load("data_prepared/X_train_flat.npy")    # Full training set
y_train = np.load("data_prepared/y_train_flat.npy")
X_val   = np.load("data_prepared/X_val_flat.npy")
y_val   = np.load("data_prepared/y_val_flat.npy")
X_test  = np.load("data_prepared/X_test_flat.npy")
y_test  = np.load("data_prepared/y_test_flat.npy")

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

# Résultats des notebooks précédents
best_info   = pd.read_csv("results/best_optuna_trial.csv")
baseline_df = pd.read_csv("results/baseline_results.csv")
all_trials  = pd.read_csv("results/optuna_trials_results.csv")

print("Meilleur hyperparamètre trouvé :")
print(best_info.to_string(index=False))
print(f"\nDataset complet : X_train={X_train.shape}, X_test={X_test.shape}")

## 2. Entraînement du modèle final optimisé

On réentraîne avec les meilleurs hyperparamètres sur le **dataset complet** (pas seulement le sous-ensemble de 10k).

In [ ]:
# Extraction des meilleurs hyperparamètres
best = best_info.iloc[0]
best_n_est    = int(best["n_estimators"])
best_depth    = None if str(best["max_depth"]) in ["None", "nan"] else int(best["max_depth"])
best_minsamp  = int(best["min_samples_split"])
best_maxfeat  = best["max_features"]
# Convertir en float si c'est un nombre
try:
    best_maxfeat = float(best_maxfeat)
except (ValueError, TypeError):
    pass

print(f"Hyperparamètres sélectionnés :")
print(f"  n_estimators     = {best_n_est}")
print(f"  max_depth        = {best_depth}")
print(f"  min_samples_split= {best_minsamp}")
print(f"  max_features     = {best_maxfeat}")

final_model = RandomForestClassifier(
    n_estimators=best_n_est,
    max_depth=best_depth,
    min_samples_split=best_minsamp,
    max_features=best_maxfeat,
    random_state=42,
    n_jobs=-1
)

# Entraînement sur dataset complet
tracemalloc.start()
t0 = time.time()
final_model.fit(X_train, y_train)
final_train_time = time.time() - t0
_, final_mem_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"\nEntraînement terminé : {final_train_time:.2f}s | Mémoire pic : {final_mem_peak/1e6:.1f} MB")

## 3. Évaluation complète du modèle final

In [ ]:
# Inférence sur test
t0 = time.time()
y_pred_final = final_model.predict(X_test)
final_inference_time = time.time() - t0
final_inf_ms = final_inference_time / len(X_test) * 1000

final_acc  = accuracy_score(y_test, y_pred_final)
final_f1   = f1_score(y_test, y_pred_final, average="weighted")

# Taille du modèle
os.makedirs("models", exist_ok=True)
model_path = "models/fashion_mnist_optimized_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(final_model, f)
final_model_size_kb = os.path.getsize(model_path) / 1024

print("=== Résultats du modèle final optimisé ===")
print(f"Accuracy test       : {final_acc:.4f} ({final_acc*100:.2f}%)")
print(f"F1-score (weighted) : {final_f1:.4f}")
print(f"Temps entraînement  : {final_train_time:.2f} s")
print(f"Inférence totale    : {final_inference_time:.4f} s ({final_inf_ms:.5f} ms/image)")
print(f"Mémoire pic         : {final_mem_peak/1e6:.1f} MB")
print(f"Taille modèle       : {final_model_size_kb:.1f} KB")
print(f"Nb paramètres (arbres) : {best_n_est} arbres")

## 4. Comparaison Baseline vs Modèle Optimisé

In [ ]:
# Récupérer les métriques du baseline Random Forest
rf_baseline_row = baseline_df[baseline_df["model_name"] == "Random Forest"].iloc[0]

comparison = pd.DataFrame([
    {
        "model": "Random Forest Baseline",
        "accuracy": rf_baseline_row["accuracy"],
        "training_time": rf_baseline_row["training_time_s"],
        "inference_ms_per_sample": rf_baseline_row["inference_ms_per_sample"],
        "model_size_kb": rf_baseline_row["model_size_kb"]
    },
    {
        "model": "Random Forest Optimisé",
        "accuracy": final_acc,
        "training_time": final_train_time,
        "inference_ms_per_sample": final_inf_ms,
        "model_size_kb": final_model_size_kb
    }
])

comparison.to_csv("results/final_comparison.csv", index=False)

# Delta
delta_acc  = (final_acc - rf_baseline_row["accuracy"]) * 100
delta_inf  = final_inf_ms - rf_baseline_row["inference_ms_per_sample"]

print("=== Comparaison Baseline vs Optimisé ===")
print(comparison[["model","accuracy","training_time","inference_ms_per_sample"]].to_string(index=False))
print(f"\nGain en accuracy     : {delta_acc:+.3f} points")
print(f"Variation inférence  : {delta_inf:+.5f} ms/image")

## 5. Visualisation de la comparaison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
labels = comparison["model"].tolist()
colors = ["#95A5A6", "#2E86AB"]

# Accuracy
axes[0].bar(labels, comparison["accuracy"], color=colors, edgecolor="white")
axes[0].set_title("Accuracy (Test Set)", fontweight="bold")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim([0.85, 1.0])
for i, v in enumerate(comparison["accuracy"]):
    axes[0].text(i, v + 0.003, f"{v:.4f}", ha="center", fontsize=10)

# Temps d'entraînement
axes[1].bar(labels, comparison["training_time"], color=colors, edgecolor="white")
axes[1].set_title("Temps d'entraînement (s)", fontweight="bold")
axes[1].set_ylabel("Secondes")
for i, v in enumerate(comparison["training_time"]):
    axes[1].text(i, v + 0.5, f"{v:.1f}s", ha="center", fontsize=10)

# Inférence
axes[2].bar(labels, comparison["inference_ms_per_sample"], color=colors, edgecolor="white")
axes[2].set_title("Inférence (ms/image)", fontweight="bold")
axes[2].set_ylabel("ms / image")
for i, v in enumerate(comparison["inference_ms_per_sample"]):
    axes[2].text(i, v + 0.0001, f"{v:.5f}", ha="center", fontsize=9)

for ax in axes:
    ax.set_xticklabels(labels, rotation=15, ha="right")

plt.suptitle("Baseline vs Random Forest Optimisé", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("results/final_comparison_plot.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Matrice de confusion et rapport de classification

In [ ]:
cm = confusion_matrix(y_test, y_pred_final)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im)
ax.set_title("Matrice de Confusion — Random Forest Optimisé", fontsize=13, fontweight="bold")
ax.set_xlabel("Classe prédite", fontsize=11)
ax.set_ylabel("Classe réelle", fontsize=11)
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(class_names, fontsize=9)
for i in range(10):
    for j in range(10):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=7,
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.tight_layout()
plt.savefig("results/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nRapport de classification complet :")
print(classification_report(y_test, y_pred_final, target_names=class_names))

## 7. Analyse critique des méthodes d'optimisation

In [ ]:
method_comp = pd.read_csv("results/method_comparison.csv")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
methods = method_comp["Méthode"].tolist()
colors  = ["#2E86AB", "#E84855", "#3BB273", "#F4A261"]

# Best accuracy par méthode
axes[0,0].bar(methods, method_comp["Best accuracy"], color=colors, edgecolor="white")
axes[0,0].set_title("Meilleure Accuracy par méthode", fontweight="bold")
axes[0,0].set_ylabel("Accuracy")
axes[0,0].set_ylim([0.80, 1.0])
for i, v in enumerate(method_comp["Best accuracy"]):
    axes[0,0].text(i, v + 0.003, f"{v:.3f}", ha="center", fontsize=9)
axes[0,0].set_xticklabels(methods, rotation=20, ha="right")

# Min inference
axes[0,1].bar(methods, method_comp["Min inférence ms"], color=colors, edgecolor="white")
axes[0,1].set_title("Meilleur Temps d'inférence par méthode", fontweight="bold")
axes[0,1].set_ylabel("ms / image")
for i, v in enumerate(method_comp["Min inférence ms"]):
    axes[0,1].text(i, v + 0.0001, f"{v:.4f}", ha="center", fontsize=9)
axes[0,1].set_xticklabels(methods, rotation=20, ha="right")

# Temps total de recherche
axes[1,0].bar(methods, method_comp["Temps recherche (s)"], color=colors, edgecolor="white")
axes[1,0].set_title("Temps total de recherche AutoML", fontweight="bold")
axes[1,0].set_ylabel("Secondes")
for i, v in enumerate(method_comp["Temps recherche (s)"]):
    axes[1,0].text(i, v + 0.5, f"{v:.1f}s", ha="center", fontsize=9)
axes[1,0].set_xticklabels(methods, rotation=20, ha="right")

# Solutions Pareto
axes[1,1].bar(methods, method_comp["Nb solutions Pareto"], color=colors, edgecolor="white")
axes[1,1].set_title("Nombre de solutions Pareto trouvées", fontweight="bold")
axes[1,1].set_ylabel("Nombre de solutions")
for i, v in enumerate(method_comp["Nb solutions Pareto"]):
    axes[1,1].text(i, v + 0.1, str(int(v)), ha="center", fontsize=11)
axes[1,1].set_xticklabels(methods, rotation=20, ha="right")

plt.suptitle("Analyse comparative des 4 méthodes d'optimisation", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("results/method_analysis_plot.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Évolution de l'accuracy au fil des trials (Convergence)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
method_colors_map = {
    "Bayesian": "#2E86AB",
    "Hyperband": "#E84855",
    "NSGA-II": "#3BB273",
    "Genetic Algorithm": "#F4A261"
}

for method, color in method_colors_map.items():
    subset = all_trials[all_trials["method"] == method].reset_index(drop=True)
    # Meilleure accuracy cumulée
    best_so_far = subset["accuracy"].cummax()
    ax.plot(range(len(subset)), best_so_far, color=color, label=method, linewidth=2, marker="o", markersize=5)

ax.set_xlabel("Numéro du trial", fontsize=11)
ax.set_ylabel("Meilleure accuracy (cumulée)", fontsize=11)
ax.set_title("Convergence des méthodes d'optimisation", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("results/convergence_plot.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Analyse de l'importance des features

In [ ]:
importances = final_model.feature_importances_.reshape(28, 28)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im = axes[0].imshow(importances, cmap="hot")
plt.colorbar(im, ax=axes[0])
axes[0].set_title("Importance des pixels (Random Forest)", fontweight="bold")
axes[0].set_xlabel("Colonne pixel")
axes[0].set_ylabel("Ligne pixel")

# Top 10 pixels les plus importants
flat_importance = final_model.feature_importances_
top10_idx = np.argsort(flat_importance)[::-1][:10]
top10_vals = flat_importance[top10_idx]
top10_labels = [f"Pixel ({i//28},{i%28})" for i in top10_idx]

axes[1].barh(range(10), top10_vals[::-1], color="#2E86AB")
axes[1].set_yticks(range(10))
axes[1].set_yticklabels(top10_labels[::-1], fontsize=9)
axes[1].set_xlabel("Importance")
axes[1].set_title("Top 10 pixels les plus importants", fontweight="bold")
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig("results/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Recommandations selon le contexte d'usage

L'optimisation multi-objectifs montre qu'il n'existe pas de solution universellement optimale. Le choix dépend du contexte déploiement.

In [ ]:
pareto_solutions = pd.read_csv("results/pareto_solutions.csv")

print("""
╔══════════════════════════════════════════════════════════════════════════╗
║           RECOMMANDATIONS PAR CONTEXTE D'USAGE                         ║
╠══════════════════════════════════════════════════════════════════════════╣
║  Contexte              │ Priorité          │ Solution recommandée       ║
╠══════════════════════════════════════════════════════════════════════════╣
║  Serveur puissant      │ Accuracy max      │ RF grand (n_est élevé)     ║
║  Application mobile    │ Inférence rapide  │ RF petit (n_est faible)    ║
║  Système embarqué      │ Taille + vitesse  │ k-NN ou RF minimal         ║
║  Prototype rapide      │ Temps recherche ↓ │ Bayesian (moins de trials) ║
╚══════════════════════════════════════════════════════════════════════════╝
""")

# Sélection des 3 meilleures solutions selon le contexte
best_accuracy  = pareto_solutions.nlargest(1, "accuracy").iloc[0]
best_inference = pareto_solutions.nsmallest(1, "inference_ms").iloc[0]
pareto_solutions["score_balance"] = (
    (pareto_solutions["accuracy"] - pareto_solutions["accuracy"].min()) /
    (pareto_solutions["accuracy"].max() - pareto_solutions["accuracy"].min() + 1e-9)
) - (
    (pareto_solutions["inference_ms"] - pareto_solutions["inference_ms"].min()) /
    (pareto_solutions["inference_ms"].max() - pareto_solutions["inference_ms"].min() + 1e-9)
)
best_balance = pareto_solutions.nlargest(1, "score_balance").iloc[0]

print("\n--- Solution 1 : MEILLEURE ACCURACY (Serveur puissant) ---")
print(f"  Accuracy: {best_accuracy['accuracy']:.4f}")
print(f"  n_estimators: {best_accuracy['n_estimators']}, max_depth: {best_accuracy['max_depth']}")
print(f"  Méthode qui l'a trouvée: {best_accuracy['method']}")

print("\n--- Solution 2 : INFÉRENCE LA PLUS RAPIDE (Mobile/Embarqué) ---")
print(f"  Inférence: {best_inference['inference_ms']:.5f} ms/image")
print(f"  Accuracy: {best_inference['accuracy']:.4f}")
print(f"  n_estimators: {best_inference['n_estimators']}, max_depth: {best_inference['max_depth']}")

print("\n--- Solution 3 : MEILLEUR COMPROMIS (Usage général) ---")
print(f"  Accuracy: {best_balance['accuracy']:.4f} | Inférence: {best_balance['inference_ms']:.5f} ms")
print(f"  n_estimators: {best_balance['n_estimators']}, max_depth: {best_balance['max_depth']}")

## 11. Tableau récapitulatif final

In [ ]:
print("="*70)
print("RÉSUMÉ DU PROJET — Optimisation Multi-Objectifs AutoML (Option 2)")
print("="*70)
print(f"\nDataset      : Fashion-MNIST (70 000 images, 10 classes)")
print(f"Modèle       : Random Forest (ML classique)")
print(f"Sous-ensemble: 10 000 images pour l'optimisation")
print(f"\nAlgorithmes comparés (20 trials chacun) :")
for _, row in method_comp.iterrows():
    print(f"  {row['Méthode']:20s} | Best acc: {row['Best accuracy']:.4f} "
          f"| Pareto: {int(row['Nb solutions Pareto'])} solutions "
          f"| Temps: {row['Temps recherche (s)']:.1f}s")
print(f"\nModèle final optimisé :")
print(f"  Accuracy test : {final_acc:.4f}")
print(f"  Inférence     : {final_inf_ms:.5f} ms/image")
print(f"  Taille        : {final_model_size_kb:.0f} KB")
print(f"  Nb arbres     : {best_n_est}")
print("\nFichiers générés :")
for fname in sorted(os.listdir("results")):
    size = os.path.getsize(f"results/{fname}") / 1024
    print(f"  results/{fname:40s} {size:.1f} KB")

print("="*70)